# 02 — HAR-RV (weekly silver volatility)

First two models on the volatility ladder:

- **Naïve** — $\hat{\text{RV}}_t = \text{RV}_{t-1}$. The floor. Volatility has a strong
  AR(1) component, so last week's RV is already a hard baseline to beat.
- **HAR-RV** (Corsi 2009) — OLS regression of RV on its short / medium / long trailing
  averages.

Features come from `volatility_weekly.csv`, built in `00_features.ipynb` — run that
notebook first.


## Setup


In [43]:
import sys, os
sys.path.append(os.path.abspath('../../src'))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from vol_utils import vol_evaluate, vol_period_metrics, vol_diebold_mariano, walk_forward
from eval_utils import PERIODS
import warnings; warnings.filterwarnings('ignore')

frame = pd.read_csv('../../data/processed/volatility_weekly.csv',
                    parse_dates=['Date']).set_index('Date')

# HAR-RV is plain OLS with no hyperparameters -> train and val are pooled into `trval`
# for the canonical fit; we do not need separate `train_df` / `val_df` here. (The RF
# and XGB notebooks in `03`/`04` keep them separate because they grid-search
# hyperparameters on val; for HAR there is nothing to tune.)
test_df  = frame[frame['split'] == 'test']
trval_df = frame[frame['split'] != 'test']

FEATS_HAR = ['rv_w_lag1', 'rv_m_lag1', 'rv_q_lag1']
print(f'train+val: {len(trval_df)}   test: {len(test_df)}')

train+val: 405   test: 174


## 1. Naïve baseline — $\hat{\text{RV}}_t = \text{RV}_{t-1}$

The naïve prediction is simply last week's RV, which is exactly the `rv_w_lag1` column.
Because volatility is highly persistent this is a genuinely strong baseline — every
other model has to beat it to justify itself.

(Its DCA is ≈ 0 by construction: predicting last week's RV implies *no* change, so the
naïve model can never call a direction. It is a floor for RMSE/MAE, not for DCA.)


In [44]:
y_test    = test_df['target'].values
prev_test = test_df['rv_w_lag1'].values        # RV_{t-1}

results = []
results.append(vol_evaluate('Naive (RV_t-1)', y_test, prev_test, prev_test))


Naive (RV_t-1)                  RMSE=0.03986  MAE=0.02014  R2=-0.094  DCA=0.000


## 2. HAR-RV (Corsi 2009)

**HAR-RV** stands for the *Heterogeneous Autoregressive model of Realised Volatility*.
It is a simple linear regression that predicts realised volatility from its own past
values measured over several horizons. The "heterogeneous" in the name refers to the
economic story behind it: different market participants operate on different time
horizons — short-term traders care about last week, medium-term traders about last
month, longer-term traders about last quarter — so the volatility observed today
reflects a mixture of all of those horizons.

Concretely it is OLS on the three HAR features built in `00_features.ipynb`. The
subscripts $w$, $m$ and $q$ stand for **week / month / quarter** — the short, medium
and long horizons of those features: $\text{RV}^{(w)}_{t-1}$ is just last week's RV
(`rv_w_lag1`), $\text{RV}^{(m)}_{t-1}$ is the 4-week trailing mean of past RV
(`rv_m_lag1`, ≈ 1 month) and $\text{RV}^{(q)}_{t-1}$ is the 12-week trailing mean
(`rv_q_lag1`, ≈ 1 quarter). All three are `.shift(1)`-ed in `00_features.ipynb` so only
information available at the close of week $t-1$ enters the week-$t$ forecast.

$$\text{RV}_t = \beta_0 + \beta_w \text{RV}^{(w)}_{t-1} + \beta_m \text{RV}^{(m)}_{t-1} + \beta_q \text{RV}^{(q)}_{t-1} + \varepsilon_t$$

Despite its simplicity HAR-RV is the standard volatility benchmark in the literature,
because those three lags capture the long-memory persistence of volatility well.
Because OLS has no hyperparameters to tune, the validation split serves no
model-selection role here — train and val are pooled for the final fit and the model
is scored once on the held-out test set. (The RF and XGB notebooks in `03`/`04` keep
`val_df` separate because they grid-search hyperparameters on it.)

In [45]:
# Single-fit on train+val for the canonical OLS coefficient table; the walk-forward
# refit below produces the predictions that feed downstream DM / period / save.
X_tr = sm.add_constant(trval_df[FEATS_HAR])
y_tr = trval_df['target']
har_fit = sm.OLS(y_tr, X_tr).fit()
print(har_fit.summary().tables[1])

# OLS wrapper for walk_forward (mimics sklearn's `.predict(X) -> array` signature).
class _OLSWrap:
    def __init__(self, fit): self.fit = fit
    def predict(self, X):    return self.fit.predict(sm.add_constant(X, has_constant='add')).values

def _ols_fit(X, y):
    return _OLSWrap(sm.OLS(y, sm.add_constant(X)).fit())

# Walk-forward HAR-RV: refit OLS every week on all data up to t-1 (expanding window).
# Mirrors GARCH's walk-forward refit and the returns chapter's walk_forward convention.
har_pred = walk_forward(frame, test_df.index, FEATS_HAR, fit_fn=_ols_fit, refit_every=1)
results.append(vol_evaluate('HAR-RV', y_test, har_pred, prev_test))

# Single-fit RMSE for reference (uses the canonical-coefficients OLS fit above).
har_pred_single = har_fit.predict(sm.add_constant(test_df[FEATS_HAR])).values
print(f'\nSingle-fit HAR-RV RMSE (for reference): '
      f'{np.sqrt(((y_test - har_pred_single)**2).mean()):.5f}    '
      f'walk-forward HAR-RV RMSE: {np.sqrt(((y_test - har_pred)**2).mean()):.5f}')

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.003      3.196      0.002       0.003       0.014
rv_w_lag1      0.0668      0.062      1.074      0.284      -0.056       0.189
rv_m_lag1      0.3971      0.126      3.155      0.002       0.150       0.644
rv_q_lag1      0.2799      0.129      2.173      0.030       0.027       0.533
HAR-RV                          RMSE=0.03205  MAE=0.01627  R2=+0.293  DCA=0.707

Single-fit HAR-RV RMSE (for reference): 0.03152    walk-forward HAR-RV RMSE: 0.03205


## 3. DM vs the Naïve floor — does HAR-RV genuinely beat $\text{RV}_{t-1}$?

The headline RMSE table above puts Naïve and HAR-RV side by side, but it does *not* say
whether the gap is statistically meaningful. This section runs the chapter's central
test — Diebold-Mariano (1995) with Newey-West (1987) lag-1 variance via
`vol_diebold_mariano` — so the "is weekly silver volatility predictable beyond last
week's value?" question is answered inside the model's own notebook, not only in the
cross-model `evaluation.ipynb`. Negative DM = HAR-RV has lower loss; stars mark
significance.

**QLIKE is the primary loss.** Weekly silver RV is heavy-tailed enough that under
squared-error loss a handful of extreme weeks (notably the 2026 spike) can carry the
majority of the loss differential, inflating the DM variance and killing the test's
power even when the RMSE gap is real and steady. The volatility-forecasting literature
(Patton 2011) therefore compares forecasts under **QLIKE**, a proxy-robust ratio-based
loss. Squared-error DM is kept below as a reference — the gap between the two stats is
itself the heavy-tail problem in action.


In [46]:
# Diebold-Mariano: HAR-RV vs the Naive RV_{t-1} floor.
print('QLIKE loss  --  primary test:')
vol_diebold_mariano(y_test, har_pred, prev_test, 'HAR-RV', 'Naive', loss='qlike')

print('\nSquared-error loss  --  reference:')
vol_diebold_mariano(y_test, har_pred, prev_test, 'HAR-RV', 'Naive', loss='mse');


QLIKE loss  --  primary test:
HAR-RV                       vs Naive         [qlike]  DM=-2.970  p=0.003  **    -> winner: HAR-RV

Squared-error loss  --  reference:
HAR-RV                       vs Naive         [mse  ]  DM=-1.260  p=0.208  (ns)  -> winner: tie


## 4. Sub-period breakdown

RMSE and DCA for HAR-RV split by calendar year, using the shared `PERIODS` definition
from `eval_utils`. This shows whether the model holds up across the choppy 2023, the
2024 bull start, the 2025 bull run, and 2026 YTD.


In [47]:
period_har = vol_period_metrics(y_test, har_pred, prev_test, test_df.index, PERIODS)
period_har.to_csv('../../data/processed/period_har_volatility.csv')
period_har.round(4)


,n,RMSE,MAE,DCA
Period,,,,
2023 (choppy),52,0.0148,0.0122,0.6923
2024 (bull start),52,0.0146,0.0115,0.7885
2025 (bull run),52,0.0222,0.0155,0.6731
2026 (YTD),18,0.0852,0.0442,0.6111
── Full test ──,174,0.0320,0.0163,0.7069


## 5. Ablations

HAR-RV uses only the silver RV's own past. This section asks three distinct questions
on top of that baseline, with every rung fitted as plain OLS:

1. **Cross-asset spillover** — do the six EXOG cross-asset RV lags (gold, copper, USD,
   S&P500, VIX, oil) carry next-week silver-volatility information beyond HAR's own
   history? `HAR+EXOG` is the full linear spillover model — the **linear sibling** of
   the RF/XGB models in `03`/`04`, so the gap between `HAR+EXOG` (here) and
   `RF/XGB (HAR+EXOG)` cleanly isolates whatever nonlinearity the trees add.
2. **Reddit sentiment** — does **public social-media sentiment** carry information
   beyond HAR? The news→volatility channel (Engle & Ng 1993), with Audrino, Sigrist &
   Ballinari (2020) as the closest precedent. Rungs separate attention from intensity
   so any effect is attributable.
3. **Paid news (NewsAPI.ai)** — do reputable-source news features (log article volume +
   FinBERT title tone intensity) carry RV information beyond the Reddit rungs? Parallel
   to Reddit: `HAR+PaidAttention`, `HAR+PaidSent`, `HAR+Paid`.

Every rung is fitted and scored on the **same sample** — the weeks where both Reddit
and paid-news features are non-missing — so the comparison is apples-to-apples.

| Rung | Adds to HAR | Tests |
|---|---|---|
| `HAR` | — | univariate baseline |
| `HAR+EXOG` | 6 cross-asset RV lags | linear cross-asset spillover |
| `HAR+RedditAttention` | `reddit_attention_lag1` | Reddit attention |
| `HAR+RedditSent` | `reddit_sent_abs_lag1` | Reddit tone intensity |
| `HAR+Reddit` | both Reddit features | Reddit combined |
| `HAR+Trends` | `trends_lag1` | Google-search retail attention |
| `HAR+PaidAttention` | `paid_attention_lag1` | paid-news volume |
| `HAR+PaidSent` | `paid_sent_abs_lag1` | paid-news tone intensity |
| `HAR+Paid` | both paid-news features | paid-news combined |

Significance is read from **QLIKE-DM** (Patton 2011) vs the `HAR` baseline —
squared-error DM is near-powerless on this heavy-tailed target (see `evaluation.ipynb`).

this cell is commented
<!-- **How the HAR formula extends.** All five rungs share the same base — the three HAR
lags from §2 — and just append additional regressors. Bare HAR is

$$\text{RV}_t \;=\; \beta_0
\;+\; \beta_w\,\text{RV}^{(w)}_{t-1}
\;+\; \beta_m\,\text{RV}^{(m)}_{t-1}
\;+\; \beta_q\,\text{RV}^{(q)}_{t-1}
\;+\; \varepsilon_t.$$

Each ablation appends a small additional linear block; OLS estimates all coefficients
jointly per rung.

- **`HAR+EXOG`** — adds the six cross-asset RV lags (gold, copper, USD, S&P 500, VIX, oil):

  $$+\ \sum_{j}\, \gamma_j\,\text{RV}^{(j)}_{t-1}.$$

- **`HAR+Attention`** — adds the Reddit attention regressor:

  $$+\ \delta_a\,\text{att}_{t-1}.$$

- **`HAR+SentIntensity`** — adds the two sentiment-intensity regressors:

  $$+\ \delta_{|s|}\,|\bar s|_{t-1}\ +\ \delta_{\sigma_s}\,\sigma_{s,\,t-1}.$$

- **`HAR+Attention+SentIntensity`** — adds all three Reddit features:

  $$+\ \delta_a\,\text{att}_{t-1}\ +\ \delta_{|s|}\,|\bar s|_{t-1}\ +\ \delta_{\sigma_s}\,\sigma_{s,\,t-1}.$$

The three Reddit features, all built on a W-FRI calendar and `.shift(1)`-ed in
`00_features` §5, are:

- **Attention** — log of weekly post volume:
  $$\text{att}_{t-1} \;=\; \log\!\Big(1 + \sum_{d \in \text{week } t-1} N_d\Big),$$
  where $N_d$ is the daily Reddit post count (`log1p` compresses the heavy right tail).

- **Sentiment magnitude** — absolute value of the weekly-mean Twitter-RoBERTa tone:
  $$|\bar s|_{t-1} \;=\; \Big|\,\tfrac{1}{|\text{week } t-1|}\sum_{d \in \text{week } t-1} s_d\,\Big|,$$
  where $s_d$ is the daily sentiment score. The sign is dropped because for
  *volatility* the direction of sentiment should not matter — only its intensity.

- **Sentiment dispersion** — within-week standard deviation of daily tone:
  $$\sigma_{s,\,t-1} \;=\; \operatorname{std}\!\big(\{s_d : d \in \text{week } t-1\}\big),$$
  a disagreement / divergence-of-opinion proxy.

Nothing changes about the OLS machinery — every rung is just
`sm.OLS(target, sm.add_constant(features)).fit()`. -->


**How the HAR formula extends — generic view.** Every HAR-X rung — sentiment,
cross-asset, or any combination — is just bare HAR with an extra linear block of
regressors. Write $x^{(k)}_{t-1}$ for the $k$-th additional feature (a cross-asset RV
lag, a Reddit attention value, a sentiment-magnitude term — whatever). A HAR-X model
with $K$ additional regressors is then

$$\text{RV}_t \;=\; \beta_0
\;+\; \sum_{h \in \{w,m,q\}} \beta_h\,\text{RV}^{(h)}_{t-1}
\;+\; \sum_{k=1}^{K} \gamma_k\,x^{(k)}_{t-1}
\;+\; \varepsilon_t.$$

The rung *name* just labels which $x^{(k)}$ are plugged in: `HAR+EXOG` uses the six
cross-asset RV lags; `HAR+RedditAttention` uses `reddit_attention_lag1`;
`HAR+RedditSent` uses the sentiment-intensity feature; `HAR+Reddit` uses both Reddit
features; `HAR+Trends` uses `trends_lag1` (Google-search attention). OLS fits the whole
$(\beta_0, \beta_w, \beta_m, \beta_q, \gamma_1, \ldots, \gamma_K)$ vector jointly per
rung — nothing else changes about the estimator.

In [48]:
FEATS_SENT_ATTN = ['reddit_attention_lag1']
FEATS_SENT_INT  = ['reddit_sent_abs_lag1', # TODO:consensus → calm, disagreement → vol (make sure this is reflected)
                #    'reddit_sent_disp_lag1'. #TODO: EDA indicates disp_lag1 is not correlated with target, confirm with me before removing and updating notebooks? todo we need to explicitly mention we only keep sent_abs_lag1 in the final model, and that this is based on EDA and correlation with target. remove all mentions of 3 variables. it's 2.
                   ]
FEATS_EXOG      = [c for c in frame.columns
                   if c.endswith('_rv_lag1') and not c.startswith('silver')]
FEATS_TRENDS    = ['trends_lag1']  # Google-search attention (§5)
FEATS_PAID_ATTN = ['paid_attention_lag1']
FEATS_PAID_INT  = ['paid_sent_abs_lag1']   # paid_sent_disp_lag1 built in 00_features; pending EDA

LADDER = {
    'HAR':                 FEATS_HAR,
    'HAR+EXOG':            FEATS_HAR + FEATS_EXOG,
    'HAR+RedditAttention': FEATS_HAR + FEATS_SENT_ATTN,
    'HAR+RedditSent':      FEATS_HAR + FEATS_SENT_INT,
    'HAR+Reddit':          FEATS_HAR + FEATS_SENT_ATTN + FEATS_SENT_INT,
    'HAR+Trends':          FEATS_HAR + FEATS_TRENDS,
    'HAR+PaidAttention':   FEATS_HAR + FEATS_PAID_ATTN,
    'HAR+PaidSent':        FEATS_HAR + FEATS_PAID_INT,
    'HAR+Paid':            FEATS_HAR + FEATS_PAID_ATTN + FEATS_PAID_INT,
}

# common sample: every rung fitted/scored on weeks where all features exist
sent_cols = FEATS_SENT_ATTN + FEATS_SENT_INT + FEATS_TRENDS + FEATS_PAID_ATTN + FEATS_PAID_INT
abl       = frame.dropna(subset=sent_cols)
abl_trval = abl[abl['split'] != 'test']
abl_test  = abl[abl['split'] == 'test']
y_abl, prev_abl = abl_test['target'].values, abl_test['rv_w_lag1'].values
print(f'ablation sample: train+val={len(abl_trval)}  test={len(abl_test)}  '
      f'(headline HAR used {len(trval_df)}/{len(test_df)})\n')

# For each rung print BOTH:
#   (a) the canonical single-fit OLS coefficient table on `abl_trval` -- gives the
#       interpretation view (coefficients, std errs, t-stats, p-values, in-sample R²);
#   (b) the walk-forward prediction RMSE/DCA -- gives the out-of-sample view that
#       feeds the DM test below.
# The walk-forward refits OLS every week; the per-week coefficients drift slightly
# from the canonical single-fit ones, but the qualitative pattern is stable.
abl_pred, abl_results = {}, []
for name, feats in LADDER.items():
    canonical = sm.OLS(abl_trval['target'], sm.add_constant(abl_trval[feats])).fit()
    print(f'=== {name}  (canonical single-fit on train+val, R²={canonical.rsquared:+.3f}) ===')
    print(canonical.summary().tables[1])

    pred = walk_forward(abl, abl_test.index, feats, fit_fn=_ols_fit, refit_every=1)
    abl_pred[name] = pred
    abl_results.append(vol_evaluate(name, y_abl, pred, prev_abl))
    print()

# QLIKE-DM: each rung vs the HAR baseline (negative DM = rung better)
print()
dm = {}
for name in LADDER:
    if name == 'HAR':
        continue
    dm[name] = vol_diebold_mariano(y_abl, abl_pred[name], abl_pred['HAR'],
                                   name, 'HAR', loss='qlike')

# Squared-error DM -- reference (mirrors §3; QLIKE is primary)
print('\nSquared-error loss  --  reference:')
dm_mse = {}
for name in LADDER:
    if name == 'HAR':
        continue
    dm_mse[name] = vol_diebold_mariano(y_abl, abl_pred[name], abl_pred['HAR'],
                                       name, 'HAR', loss='mse')

abl_df = pd.DataFrame(abl_results)
abl_df['dm_qlike']   = abl_df['model'].map(lambda m: dm[m]['dm']     if m in dm     else np.nan)
abl_df['dm_qlike_p'] = abl_df['model'].map(lambda m: dm[m]['p']      if m in dm     else np.nan)
abl_df['dm_mse']     = abl_df['model'].map(lambda m: dm_mse[m]['dm'] if m in dm_mse else np.nan)
abl_df['dm_mse_p']   = abl_df['model'].map(lambda m: dm_mse[m]['p']  if m in dm_mse else np.nan)
abl_df['winner_dm_qlike'] = abl_df['model'].map(lambda m: dm[m]['winner']     if m in dm     else np.nan)
abl_df['winner_dm_mse']   = abl_df['model'].map(lambda m: dm_mse[m]['winner'] if m in dm_mse else np.nan)
abl_df.to_csv('../../data/processed/metrics_har_sentiment_volatility.csv', index=False)
print('\nSaved metrics_har_sentiment_volatility.csv')
abl_df.round(5)

ablation sample: train+val=402  test=174  (headline HAR used 405/174)

=== HAR  (canonical single-fit on train+val, R²=+0.222) ===
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.003      3.172      0.002       0.003       0.014
rv_w_lag1      0.0699      0.063      1.114      0.266      -0.053       0.193
rv_m_lag1      0.3956      0.127      3.117      0.002       0.146       0.645
rv_q_lag1      0.2778      0.130      2.139      0.033       0.023       0.533
HAR                             RMSE=0.03205  MAE=0.01627  R2=+0.293  DCA=0.707

=== HAR+EXOG  (canonical single-fit on train+val, R²=+0.270) ===
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0066      0.004      1.649      0.100      -0.001       0.014


,model,rmse,mae,r2,dca,dm_qlike,dm_qlike_p,dm_mse,dm_mse_p,winner_dm_qlike,winner_dm_mse
0,HAR,0.03205,0.01627,0.29263,0.70690,NaN,NaN,NaN,NaN,NaN,NaN
1,HAR+EXOG,0.03268,0.01655,0.26440,0.71839,2.22996,0.02575,2.15647,0.03105,HAR,HAR
2,HAR+RedditAttention,0.03191,0.01621,0.29859,0.70690,-2.59629,0.00942,-1.92014,0.05484,HAR+RedditAttention,tie
3,HAR+RedditSent,0.03195,0.01619,0.29709,0.71264,-3.36126,0.00078,-2.60599,0.00916,HAR+RedditSent,HAR+RedditSent
4,HAR+Reddit,0.03191,0.01619,0.29889,0.71839,-3.00798,0.00263,-2.20366,0.02755,HAR+Reddit,HAR+Reddit
5,HAR+Trends,0.03156,0.01597,0.31390,0.70115,-0.01353,0.98920,-1.89732,0.05779,tie,tie
6,HAR+PaidAttention,0.03172,0.01605,0.30723,0.71839,-3.32539,0.00088,-2.40836,0.01602,HAR+PaidAttention,HAR+PaidAttention
7,HAR+PaidSent,0.03205,0.01630,0.29258,0.70115,1.43523,0.15122,0.05828,0.95352,tie,tie
8,HAR+Paid,0.03164,0.01608,0.31072,0.71839,-2.68451,0.00726,-2.17198,0.02986,HAR+Paid,HAR+Paid


**Reading the table.** `dm_qlike` / `dm_qlike_p` give the QLIKE-DM stat and p-value of
each rung vs the `HAR` baseline (negative = the rung is better; the `HAR` row is blank).
`dm_mse` / `dm_mse_p` repeat the test under squared-error loss (reference only — QLIKE is primary for this heavy-tailed target).
Three questions:

1. **Does linear cross-asset spillover beat bare HAR?** Read the `HAR+EXOG` row. A
   significantly-negative `dm_qlike` would mean the six cross-asset RVs carry linear
   silver-volatility information beyond HAR's own history. (Here the result goes the
   opposite way — `HAR+EXOG` is *significantly worse* than `HAR`, OLS overfits the six
   noisy regressors on ~400 weeks.)
2. **Does Reddit sentiment beat bare HAR?** A negative, significant `dm_qlike` on
   `HAR+RedditAttention`, `HAR+RedditSent` or the combined `HAR+Reddit` rung says yes.
   The split between the two individually-significant single-mechanism rungs and the
   combined-rung loss of significance is itself informative — see `notes.md` for the
   "three views of the same latent Reddit engagement level" explanation.
3. **Does Google-search attention beat bare HAR?** `HAR+Trends` — *no* under QLIKE,
   even though `trends_lag1` has the **highest RV correlation and the best RMSE of any
   rung**. The two losses disagree by construction. Its RMSE win comes from a *handful*
   of the highest-RV weeks (the 2026 spike), where Trends cuts the **absolute** error —
   and squared error lets those few weeks dominate (drop 2026 and the edge largely
   vanishes). In **proportional** (QLIKE) terms it only sharpens the many calm weeks and
   is *worse* across the active ones, so the two cancel and it nets to ~zero — while
   Reddit attention, whose benefit concentrates in the turbulent weeks, survives QLIKE.
   This is the clearest case in the chapter for why **QLIKE, not RMSE/R², is reported as
   the primary metric** (RMSE here is hostage to ~a dozen weeks).

A null result here is still thesis-relevant: *"sentiment adds nothing once the HAR
lags are in"* is clean semi-strong-form evidence, consistent with the returns side of
the thesis. Likewise, the gap between `HAR+EXOG` (linear) here and
`RF/XGB (HAR+EXOG)` in `03`/`04` measures **only** what nonlinearity buys on top of
linear cross-asset spillover.

## 6. Save outputs

- `metrics_har_volatility.csv` — Naïve + HAR-RV headline metrics
- `pred_har_volatility.csv` — test-set predictions, consumed by `evaluation.ipynb`
- `metrics_har_sentiment_volatility.csv` — the §5 sentiment-ablation table (saved above)

In [49]:
pd.DataFrame(results).to_csv('../../data/processed/metrics_har_volatility.csv', index=False)

pred_har = pd.DataFrame({'actual': y_test, 'prev': prev_test,
                         'naive': prev_test, 'har': har_pred}, index=test_df.index)
pred_har.to_csv('../../data/processed/pred_har_volatility.csv', index_label='Date')
print('Saved metrics_har_volatility.csv + pred_har_volatility.csv')
pd.DataFrame(results).round(5)


Saved metrics_har_volatility.csv + pred_har_volatility.csv


,model,rmse,mae,r2,dca
0,Naive (RV_t-1),0.03986,0.02014,-0.09429,0.0000
1,HAR-RV,0.03205,0.01627,0.29279,0.7069
